In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths
from metab_processing.LinearRegression.build_x import get_gene_factors

In [ ]:
paths = dataset_paths('Primary_Dermal_Melanoma', data_dir=f'{DATA_DIR}/Xenium')
adata = sc.read_h5ad(paths['dataset_dir'] / 'LinearRegression' / 'x_adata.h5ad')
ANNOT_COL = 'Tier1'
print(adata.n_obs, 'cells |', len(adata.uns['x_genes']), 'genes')

In [ ]:
def build_matrix(gene, metabs=None, annot_col=None, annot_value=None):
    """z-scored factor matrix (cells x factors) for `gene` (each column mean 0, std 1).
    `metabs`: None (default, no metabolites) / 'all' / a list. Optional cell filter:
    `annot_col` == `annot_value` (a single value or a list); default = all cells."""
    X = get_gene_factors(adata, gene, metabs=metabs)
    if annot_col is not None:
        vals = [annot_value] if isinstance(annot_value, str) else list(annot_value)
        X = X.loc[adata.obs[annot_col].astype(str).isin([str(v) for v in vals]).values]
    X = X.loc[:, X.std() > 0]          # drop constant columns (can't z-score / cluster)
    return (X - X.mean()) / X.std()

In [ ]:
def cluster_factors(Z, threshold=0.5, method='average'):
    """Hierarchical clustering of the factors. Distance = 1 - correlation, where the
    correlation of the z-scored columns IS their covariance (~ normalized dot product).
    Returns ({group_id: [factors]}, linkage)."""
    corr = np.corrcoef(Z.values.T)
    dist = 1 - corr
    np.fill_diagonal(dist, 0)
    L = linkage(squareform(dist, checks=False), method=method)
    labels = fcluster(L, t=threshold, criterion='distance')
    groups = {}
    for f, lab in zip(Z.columns, labels):
        groups.setdefault(int(lab), []).append(f)
    return groups, L

In [ ]:
GENE = adata.uns['x_genes'][0]     # <- select the target gene
Z = build_matrix(GENE, metabs=None, annot_col=ANNOT_COL, annot_value='T Cell')
groups, L = cluster_factors(Z, threshold=0.5)
print(f'{GENE}: {Z.shape[1]} factors -> {len(groups)} groups')
for g, m in sorted(groups.items()):
    print(f'group {g} ({len(m)}): {m}')

In [ ]:
def pca_group(group, Z=None, n=None):
    """PCA on one group's factors. `group`: a group id from `groups`, a list of factor
    names, or a DataFrame. Plots the explained-variance scree + a PC x factor loadings
    heatmap. Returns the fitted PCA."""
    Z = globals()['Z'] if Z is None else Z
    cols = groups[group] if isinstance(group, int) else (
        list(group.columns) if isinstance(group, pd.DataFrame) else list(group))
    p = PCA(n_components=n).fit(Z[cols].values)
    plt.plot(np.arange(1, p.n_components_ + 1), p.explained_variance_ratio_, 'o-')
    plt.xlabel('PC'); plt.ylabel('explained var'); plt.show()
    load = pd.DataFrame(p.components_, columns=cols,
                        index=[f'PC{i+1}' for i in range(p.n_components_)])
    v = np.abs(load.values).max()
    plt.figure(figsize=(min(1 + 0.3 * len(cols), 20), 1 + 0.3 * len(load)))
    plt.imshow(load, aspect='auto', cmap='RdBu_r', vmin=-v, vmax=v); plt.colorbar()
    plt.yticks(range(len(load)), load.index); plt.xticks(range(len(cols)), cols, rotation=90)
    plt.tight_layout(); plt.show()
    return p

In [ ]:
p = pca_group(sorted(groups)[0])   # a group id, or pass a list of factors / a DataFrame